# Monte Carlo Tree Search (MCTS) Implementation

In [7]:
from algorithms.mcts.MCTS import MCTS
from mdp.base_config import BaseConfig
from algorithms.mcts.PersistentMCTS import PersistentMCTS
from mdp.mdp_utils import display_value_function, display_policy, max_norm_error
from algorithms.value_iteration.value_iteration import ValueIteration

%load_ext autoreload
%autoreload 2

base_config = BaseConfig(seed=42)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Cats vs Monsters MDP

### Generate Value Function using Value Iteration

In [4]:
from mdp.cats_v_monsters.cats_vs_monsters import CatsVMonstersMDP

cvm = CatsVMonstersMDP()
optimal_vi = ValueIteration(mdp=optimal_mdp)
optimal_rh = optimal_vi.run_value_iteration()
optimal_vf, optimal_policy = optimal_rh[len(optimal_rh)]
print("Optimal Value Function:")
display_value_function(optimal_mdp, optimal_vf)
print("Optimal Policy:")
display_policy(cvm, optimal_policy)

Optimal Value Function:
2.6638	2.9969	2.8117	3.6671	4.8497	
2.9713	3.5101	4.0819	4.8497	7.1648	
2.5936	0.0000	0.0000	0.0000	8.4687	
2.0992	1.0849	0.0000	8.6097	9.5269	
1.0849	4.9465	8.4687	9.5269	0.0000	
Optimal Policy:
→	↓	←	↓	↓	
→	→	→	→	↓	
↑	F	F	F	↓	
↑	←	F	↓	↓	
↑	→	→	→	G	


In [ ]:
cvm = CatsVMonstersMDP()
mcts = MCTS(env=cvm, base_config=base_config)
root_state = (3, 4)  # Starting state with 0 cats and 0 monsters
root, action = mcts.search(root_state=root_state, num_simulations=500)
print(f'From root state {root_state}, selected action: {action}, estimated value: {root.value}')

From root state (3, 4), selected action: left, estimated value: -129.39999999999986


In [41]:
for action, child in root.children.items():
    print(f'Action: {action}, Q-value: {child.value}, Visit Count: {child.visits}')

Action: right, Q-value: -42.699999999999974, Visit Count: 1
Action: left, Q-value: 10.0, Visit Count: 249
Action: down, Q-value: 10.0, Visit Count: 249
Action: up, Q-value: -106.69999999999987, Visit Count: 1


In [43]:
value_function, policy = mcts.generate_value_function_and_policy(num_simulations=50000)

Evaluating all states:   0%|          | 0/21 [00:00<?, ?it/s]

In [44]:
display_value_function(value_function=value_function, mdp=cvm)

-0.4358	-0.2322	-0.3216	-0.2148	-0.4708	
-0.1570	-2.0820	-0.0765	-0.1203	-0.0550	
-0.5661	0.0000	0.0000	0.0000	-0.0578	
-1.1632	-0.1488	0.0000	-0.0126	-0.0024	
-0.3315	-0.0377	-0.0339	-0.0274	0.0000	


In [46]:
display_policy(cvm, policy=policy)

↑	↓	→	→	↓	
↓	←	→	↓	←	
↓	X	X	X	→	
←	↓	X	→	↓	
↑	→	↓	↑	G	


## Generating for multiple c-values

In [9]:
c_vals = [0, 0.1, 0.5, 1.0, 3.0, 5.0]
c_vals_metadata = {}
for c in c_vals:
    print(f'\nExploration constant c = {c}')
    mcts = MCTS(env=cvm, base_config=base_config)
    value_function, policy = mcts.generate_value_function_and_policy(num_simulations=5000, c_value=c)
    max_norm_error = max_norm_error(optimal_vf, value_function)
    print(f'Max Norm Error compared to Optimal VI: {max_norm_error}')
    c_vals_metadata[c] = {
        'value_function': value_function,
        'policy': policy,
        'max_norm_error': max_norm_error
    }


Exploration constant c = 0


Evaluating all states:   0%|          | 0/21 [00:00<?, ?it/s]

KeyError: (2, 1)